# Stage 06: Data Preprocessing Submission

This notebook applies reusable median imputation, missingness-based column removal, and min-max normalization to the provided raw dataset.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data

RAW_PATH = PROJECT_ROOT / "data/raw/sample_data.csv"
PROCESSED_PATH = PROJECT_ROOT / "data/processed/sample_data_cleaned.csv"
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Raw data:", RAW_PATH.relative_to(PROJECT_ROOT))
print("Processed data:", PROCESSED_PATH.relative_to(PROJECT_ROOT))

Raw data: data/raw/sample_data.csv
Processed data: data/processed/sample_data_cleaned.csv


## 1. Load and Inspect the Raw Dataset

In [2]:
original_df = pd.read_csv(RAW_PATH, dtype={"zipcode": "string"})
print("Original shape:", original_df.shape)
print("Missing values by column:")
print(original_df.isna().sum())
display(original_df)

Original shape: (7, 6)
Missing values by column:
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## 2. Cleaning Assumptions

- `age`, `income`, and `score` are continuous numeric features; their missing values are replaced by medians to reduce sensitivity to extreme values.
- A column with more than 50% missing values is too sparse for this small exercise and is removed. This rule drops `extra_data` but keeps all seven observations.
- The three analysis features are min-max normalized to `[0, 1]`.
- `zipcode` is an identifier and is loaded as text. It and `city` are not imputed or normalized.
- Cleaning functions return copies so the raw DataFrame remains available for comparison.

In [3]:
numeric_features = ["age", "income", "score"]
imputed_df = fill_missing_median(original_df, numeric_features)
reduced_df = drop_missing(imputed_df, threshold=0.5)
cleaned_df = normalize_data(reduced_df, numeric_features)

print("Columns removed:", sorted(set(original_df.columns) - set(cleaned_df.columns)))
print("Cleaned shape:", cleaned_df.shape)
print("Remaining missing values:", int(cleaned_df.isna().sum().sum()))
display(cleaned_df)

Columns removed: ['extra_data']
Cleaned shape: (7, 5)
Remaining missing values: 0


,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin
5,0.500000,0.6250,0.000000,12345,Unknown
6,0.571429,0.4375,0.538462,94105,San Francisco


## 3. Compare Original and Cleaned Data

In [4]:
comparison = pd.DataFrame({
    "original_missing": original_df.isna().sum(),
    "cleaned_missing": cleaned_df.isna().sum(),
    "original_dtype": original_df.dtypes.astype(str),
    "cleaned_dtype": cleaned_df.dtypes.astype(str),
}).fillna({"cleaned_missing": "column removed", "cleaned_dtype": "column removed"})
display(comparison)

range_comparison = pd.concat({
    "original": original_df[numeric_features].agg(["min", "median", "max"]),
    "cleaned": cleaned_df[numeric_features].agg(["min", "median", "max"]),
}, axis=1)
display(range_comparison)

,original_missing,cleaned_missing,original_dtype,cleaned_dtype
age,1,0.0,float64,float64
city,0,0.0,str,str
extra_data,5,column removed,float64,column removed
income,3,0.0,float64,float64
score,1,0.0,float64,float64
zipcode,0,0.0,string,string


original                 cleaned                 
            age   income  score     age income     score
min        29.0  42000.0  0.650     0.0  0.000  0.000000
median     39.5  52000.0  0.805     0.5  0.625  0.596154
max        50.0  58000.0  0.910     1.0  1.000  1.000000

## 4. Validate and Save the Processed Dataset

In [5]:
assert cleaned_df.shape == (7, 5)
assert "extra_data" not in cleaned_df.columns
assert cleaned_df.isna().sum().sum() == 0
assert cleaned_df[numeric_features].min().eq(0).all()
assert cleaned_df[numeric_features].max().eq(1).all()
assert cleaned_df["zipcode"].dtype.name == "string"

cleaned_df.to_csv(PROCESSED_PATH, index=False)
reloaded_df = pd.read_csv(PROCESSED_PATH, dtype={"zipcode": "string"})
assert reloaded_df.shape == cleaned_df.shape
assert list(reloaded_df.columns) == list(cleaned_df.columns)
print("All validation checks passed: True")
print("Saved:", PROCESSED_PATH.relative_to(PROJECT_ROOT))

All validation checks passed: True
Saved: data/processed/sample_data_cleaned.csv


## Reflection and Tradeoffs

Median imputation retains every row, which is valuable for this seven-row dataset, but it makes several records look more typical than the unknown values may truly be and can understate variability. Dropping `extra_data` avoids inventing values for a column with about 71% missingness, but it may discard a potentially useful signal if more observations become available later. Min-max scaling makes the numeric features comparable for many models, yet it depends on the current sample's extremes and would need a fitted training-data scaler in a production workflow to prevent leakage. The raw CSV is preserved unchanged so these decisions remain reversible.